In [0]:
%run ./variables

In [0]:
import pyspark.sql.functions as f

In [0]:
def save_archive(df, target_table: str, run_date, merge_schema = False):
    df_w_run_date = df.withColumn("run_date", f.lit(run_date).cast('date'))
    df_w_run_date.write.option("mergeSchema", merge_schema).mode("overwrite").option("replaceWhere", f"run_date = '{run_date}'").saveAsTable(target_table)

def save_recent(df, target_table: str, merge_schema = False):
    df.write.option("mergeSchema", merge_schema).mode("overwrite").saveAsTable(target_table)


def save_recent_archive(df, recent_target_table: str, archive_target_table: str, run_date, merge_schema = False):
    
    save_archive(df, archive_target_table, run_date, merge_schema)
    save_recent(df, recent_target_table, merge_schema)

In [0]:
def save_cf_tables(df, name, params):
    df = df.withColumn("CATEGORY_CD", f.lit(params["category"]))
    df = df.withColumn("START_DATE", f.to_date(f.lit(params["start"]), "yyyy-MM-dd"))
    df = df.withColumn("END_DATE", f.to_date(f.lit(params["end"]), "yyyy-MM-dd"))
    df = df.withColumn("RUN_NAME", f.to_date(f.lit(params["run_name"][-10:]), "yyyy_MM_dd"))
    
    df.write.mode('overwrite').option('replaceWhere', f"CATEGORY_CD = '{params['category']}' AND START_DATE = '{params['start']}' AND END_DATE = '{params['end']}' AND RUN_NAME = '{params['run_name'][-10:].replace('_','-')}'").saveAsTable(name)

In [0]:
def read_cf_tables(name, params, past_flag=False, future_flag=False, combined=False):
      if past_flag and future_flag:
            raise ValueError("Both past and future flags cannot be set to True")
      if past_flag:
            start = params["p_start"]
            end = params["p_end"]
      elif future_flag:
            start = params["f_start"]
            end = params["f_end"]
      else:
            start = params["start"]
            end = params["end"]
      if combined:
            df = (spark.read.table(name)
                        .filter(f.col("START_DATE") == f.to_date(f.lit(start), "yyyy-MM-dd"))
                        .filter(f.col("END_DATE") == f.to_date(f.lit(end), "yyyy-MM-dd"))
                        .filter(f.col("RUN_NAME") == f.to_date(f.lit(params["run_name"][-10:]), "yyyy_MM_dd"))
                  )
      else:
            df = (spark.read.table(name)
                  .filter(f.col("CATEGORY_CD") == params["category"])
                  .filter(f.col("START_DATE") == f.to_date(f.lit(start), "yyyy-MM-dd"))
                  .filter(f.col("END_DATE") == f.to_date(f.lit(end), "yyyy-MM-dd"))
                  .filter(f.col("RUN_NAME") == f.to_date(f.lit(params["run_name"][-10:]), "yyyy_MM_dd"))
                  )
      return df.drop("CATEGORY_CD", "START_DATE", "END_DATE", "RUN_NAME")